# PRISM-3D on Google Colab: pilot, full study, tables and figures

Run the cells top to bottom. Every stage is resumable: if the runtime disconnects, re-run the setup cells and then the stage you were on; finished images are skipped.

**Runtime:** GPU (T4 is enough for the pilot; L4 or A100 recommended for the full study).
**Storage:** results are written to Google Drive so nothing is lost when the runtime resets.

Order: setup, tests, smoke test, downloads, model checks, **pilot (stop here and read the decision)**, full study, ablations, tables, figures.

In [ ]:
!nvidia-smi

## 1. Setup: Drive, code, packages

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = '/content/drive/MyDrive/prism3d_work'     # results, data and third-party code persist here
os.makedirs(WORK, exist_ok=True)

In [ ]:
# Option A: upload prism3d.zip to your Drive first, then unzip it here
!unzip -q -o /content/drive/MyDrive/prism3d.zip -d /content/
%cd /content/prism3d
# keep heavy folders on Drive (symlinks), so downloads and results survive a reset
for d in ['data/raw', 'third_party', 'results']:
    os.makedirs(f'{WORK}/{d}', exist_ok=True)
    !rm -rf {d} && mkdir -p $(dirname {d}) && ln -s {WORK}/{d} {d}
!ls -la

In [ ]:
!pip install -q -r requirements.txt
!pip install -q git+https://github.com/microsoft/MoGe.git
!pip install -q https://github.com/compphoto/Intrinsic/archive/main.zip

## 2. Unit tests and smoke test (CPU, about 3 minutes)
Both must pass before using GPU time. The smoke test runs every stage on synthetic renders.

In [ ]:
!python -m unittest discover -s tests -t .
!python scripts/run_smoke.py

## 3. Downloads
Read the dry run first. Chameleon needs an access request (see docs/dataset_guide.md).

In [ ]:
!python scripts/download_data.py all --dry-run

In [ ]:
!python scripts/download_data.py detectors
!python scripts/download_data.py synthbuster synthbuster_ext
!python scripts/download_data.py cospy flickr30k
!python scripts/download_data.py openfake --openfake-max-per-model 600
!python scripts/download_data.py chameleon   # prints the manual steps

## 4. Model checks on one image
Confirm shapes and that predicted normals and depth normals look alike on a real photo.

In [ ]:
import glob
img = sorted(glob.glob('data/raw/synthbuster_ext/**/*.*', recursive=True))[0]
!python scripts/check_estimators.py --image "{img}"
from IPython.display import Image as _I; _I('results/check_estimators/panel.png')

## 5. Pilot (pre-registered GO / REVISE / STOP)
UnivFD base, clean and JPEG 75 copies. Read the decision before running the full study.

In [ ]:
CFG = 'configs/pilot.yaml'
!python scripts/prepare_data.py --config {CFG}
!python scripts/run_detectors.py --config {CFG} --check      # sign check: AI images should have higher logits

In [ ]:
!python scripts/extract_geometry.py --config {CFG}
!python scripts/run_detectors.py --config {CFG}
!python scripts/extract_controls.py --config {CFG}
!python scripts/run_experiments.py --config {CFG}
!python scripts/run_ablations.py --config {CFG}
!python scripts/make_tables.py --config {CFG}

In [ ]:
import json
h = json.load(open('results/pilot/hypotheses.json'))
print(json.dumps(h['pilot'], indent=2))

**Decision rule.** GO: run the full study. REVISE: check which cue carries the signal (ablations), then repeat the pilot with all four cue groups on B-Free. STOP: write up the negative result (docs/research_sop.md, step 9).
Record the decision in docs/experiment_log.md before continuing.

## 6. Full study

In [ ]:
CFG = 'configs/default.yaml'
# If Chameleon is not available yet, append: --set data.skip_missing_sources=true
!python scripts/prepare_data.py --config {CFG}
!python scripts/run_detectors.py --config {CFG} --check

In [ ]:
!python scripts/extract_geometry.py --config {CFG}


In [ ]:
!python scripts/run_detectors.py --config {CFG}
!python scripts/extract_controls.py --config {CFG}
!python scripts/run_experiments.py --config {CFG}


### Estimator sensitivity ablation (optional but planned): Marigold normals and Retinex shading

In [ ]:
!python scripts/extract_geometry.py --config {CFG} --variants clean --feature-dir features_marigold --set geometry.normal_estimator=marigold
!python scripts/extract_geometry.py --config {CFG} --variants clean --feature-dir features_retinex --set geometry.shading_estimator=retinex


In [ ]:
!python scripts/run_ablations.py --config {CFG} --set "geometry.alternative_feature_dirs=[features_marigold, features_retinex]"
!python scripts/make_tables.py --config {CFG}
!python scripts/generate_figures.py --config {CFG}

### Optional outside reference: zero-shot Qwen2.5-VL-7B (needs about 24 GB GPU)

In [ ]:
# !python scripts/zero_shot_mllm.py --config {CFG} --limit 2000

## 7. Put results into the paper
Copy the figures into `paper/figures/` (this replaces the yellow placeholder figures). Then `fill_results.py` fills every `[xyz]` it can compute and writes `results/full/tables/paper_values.md`, a list of every key with its value, source file and line in the paper (see `docs/filling_results.md`).
If you edited `main.tex` in Overleaf, upload your version to `paper/` first. Score the scene audit before this cell if you have labelled it.

In [ ]:
!for f in results/full/figures/*.pdf results/full/figures/*.png; do b=$(basename $f); cp $f paper/figures/$b; rm -f paper/figures/${b%.*}.placeholder; done
# !python scripts/export_scene_audit.py --config {CFG} --score results/full/scene_audit.csv
!python scripts/fill_results.py --config {CFG} --pilot-config configs/pilot.yaml
!head -n 40 results/full/tables/paper_values.md
# after checking paper/main_filled.tex and paper_values.md:
# !python scripts/fill_results.py --config {CFG} --pilot-config configs/pilot.yaml --inplace && python scripts/build_docx.py
!python scripts/check_paper.py
!cp -r paper {WORK}/paper_filled

## End of a session
Run this before closing Colab so every file reaches Drive.

In [ ]:
# End of every session: make sure everything is written to Drive
from google.colab import drive
drive.flush_and_unmount()
